In [ ]:
from huggingface_hub import InferenceClient
import json, os

client = InferenceClient(
    model="Qwen/Qwen3-VL-8B-Instruct:novita",
    api_key=os.environ["HF_TOKEN"]
)

weather_data = {
    "Dublin": "20 degrees",
    "Galway": "15 degrees"
}

def weather(loc):
    return weather_data[loc]

tools = [{
    "type": "function",
    "function": {
        "name": "weather",
        "description": "Get temperature",
        "parameters": {
            "type": "object",
            "properties": {
                "loc": {"type": "string"}
            },
            "required": ["loc"]
        }
    }
}]

messages = [
    {"role": "user", "content": "weather in Galway?"}
]

resp = client.chat_completion(
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

msg = resp.choices[0].message

# tool call
if msg.tool_calls:
    tc = msg.tool_calls[0]
    args = json.loads(tc.function.arguments)

    result = weather(**args)

    messages.append(msg)
    messages.append({
        "role": "tool",
        "name": tc.function.name,
        "content": result
    })

    final = client.chat_completion(messages)
    print(final.choices[0].message.content)

It’s currently 15°C in Galway. The weather seems mild and pleasant — perfect for a walk or outdoor activity! 🌞

If you’re planning anything, feel free to ask for more details like wind speed, humidity, or forecasts for the next few days.
